# Phase 04 - GNN Model

> Message passing from scratch in PyTorch, heterogeneous graph handling, and a GAT/HGT-style attention mechanism.

This phase defines the model architecture only. Phase 05 will handle the training loop, class weighting, optimizer, checkpoints, and metrics.

The model loads the fixed Phase 03 graph from `hetero_data.pt`, uses PyTorch Geometric's `HeteroData` only as a data container, and implements the graph neural network layers manually in PyTorch.

## 4.1 - Imports and Load the Graph

We load the leakage-safe heterogeneous graph created in Phase 03. The notebook supports both Colab-style `/content/sample_data` and local `content/sample_data` paths.

In [ ]:
import math
import os
import random
import warnings
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from torch_geometric.data import HeteroData
except ImportError as exc:
    raise ImportError(
        'PyTorch Geometric is required to load the HeteroData object from Phase 03. '
        'The GNN layers below are implemented from scratch in PyTorch.'
    ) from exc

warnings.filterwarnings('ignore')

DATA_DIR = Path('/content/sample_data')
if not DATA_DIR.exists():
    DATA_DIR = Path('content/sample_data')

GRAPH_PATH = DATA_DIR / 'hetero_data.pt'
if not GRAPH_PATH.exists():
    raise FileNotFoundError(
        f'{GRAPH_PATH} not found. Run Phase03_Graph_Construction.ipynb first.'
    )

try:
    data = torch.load(GRAPH_PATH, map_location='cpu', weights_only=False)
except TypeError:
    data = torch.load(GRAPH_PATH, map_location='cpu')

# print(f'Loaded graph from: {GRAPH_PATH}')
# print(data)
# print()
# print(f'Node types : {data.node_types}')
# print(f'Edge types : {len(data.edge_types)}')

## 4.2 - Reproducibility and Device

The model is deterministic enough for experimentation, while still allowing GPU acceleration when available.

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cpu


## 4.3 - Graph Summary

Only transaction nodes have labels and split masks. Entity nodes provide relational context through their connections to transactions.

In [3]:
print('Node feature shapes:')
for node_type in data.node_types:
    x = data[node_type].x
    print(f'  {node_type:<15} {tuple(x.shape)}')

print()
print('Transaction splits:')
for split in ['train', 'val', 'test']:
    mask = data['transaction'][f'{split}_mask']
    y_split = data['transaction'].y[mask]
    fraud_rate = y_split.float().mean().item() * 100
    print(f'  {split:<5} nodes: {mask.sum().item():>8,} | fraud rate: {fraud_rate:>5.2f}%')

print()
print('Relations:')
for edge_type in data.edge_types[:8]:
    print(f'  {edge_type}: {tuple(data[edge_type].edge_index.shape)}')
if len(data.edge_types) > 8:
    print(f'  ... {len(data.edge_types) - 8} more relations')

Node feature shapes:
  transaction     (590540, 45)
  product         (6, 2)
  card1           (6513, 2)
  card2           (501, 2)
  card3           (77, 2)
  card4           (6, 2)
  card5           (77, 2)
  card6           (6, 2)
  addr1           (134, 2)
  addr2           (31, 2)
  email_p         (61, 2)
  email_r         (62, 2)
  device_type     (4, 2)
  device_info     (889, 2)

Transaction splits:
  train nodes:  413,378 | fraud rate:  3.50%
  val   nodes:   88,581 | fraud rate:  3.50%
  test  nodes:   88,581 | fraud rate:  3.50%

Relations:
  ('transaction', 'has_product', 'product'): (2, 590540)
  ('product', 'rev_has_product', 'transaction'): (2, 590540)
  ('transaction', 'has_card1', 'card1'): (2, 590540)
  ('card1', 'rev_has_card1', 'transaction'): (2, 590540)
  ('transaction', 'has_card2', 'card2'): (2, 590540)
  ('card2', 'rev_has_card2', 'transaction'): (2, 590540)
  ('transaction', 'has_card3', 'card3'): (2, 590540)
  ('card3', 'rev_has_card3', 'transaction'): (2, 5

## 4.4 - Model Configuration

The architecture uses:

- type-specific input encoders, because transaction and entity nodes have different feature dimensions
- heterogeneous attention layers, with relation-specific query/key/value projections
- transaction-only classification head for fraud prediction

The defaults are intentionally moderate. Full-batch heterogeneous attention can be memory-heavy on this graph, so Phase 05 can tune these values based on runtime memory.

In [4]:
MODEL_CONFIG = {
    'hidden_dim': 64,
    'num_heads': 4,
    'num_layers': 2,
    'dropout': 0.25,
    'num_classes': 2,
}

assert MODEL_CONFIG['hidden_dim'] % MODEL_CONFIG['num_heads'] == 0
MODEL_CONFIG

{'hidden_dim': 64,
 'num_heads': 4,
 'num_layers': 2,
 'dropout': 0.25,
 'num_classes': 2}

## 4.5 - Utility Functions

These helpers keep the code independent from PyG convolution layers. The important piece is `edge_softmax`: it computes an attention softmax over incoming edges for each destination node.

In [5]:
def move_hetero_data(data, device):
    """Move node features, labels, masks, and edge indices to the target device."""
    for node_type in data.node_types:
        store = data[node_type]
        for key, value in list(store.items()):
            if torch.is_tensor(value):
                store[key] = value.to(device)

    for edge_type in data.edge_types:
        store = data[edge_type]
        for key, value in list(store.items()):
            if torch.is_tensor(value):
                store[key] = value.to(device)
    return data


def edge_softmax(scores, dst_index, num_dst_nodes):
    """Softmax over edges grouped by destination node.

    Args:
        scores: Tensor of shape [num_edges, num_heads].
        dst_index: Destination node index for each edge, shape [num_edges].
        num_dst_nodes: Number of destination nodes for this relation.
    """
    if scores.numel() == 0:
        return scores

    num_heads = scores.size(1)
    expanded_dst = dst_index.view(-1, 1).expand(-1, num_heads)

    max_per_dst = scores.new_full((num_dst_nodes, num_heads), -torch.inf)
    max_per_dst.scatter_reduce_(0, expanded_dst, scores, reduce='amax', include_self=True)

    shifted = scores - max_per_dst[dst_index]
    exp_scores = torch.exp(shifted)

    denom = scores.new_zeros((num_dst_nodes, num_heads))
    denom.scatter_add_(0, expanded_dst, exp_scores)

    return exp_scores / (denom[dst_index] + 1e-12)


def clone_with_limited_edges(data, max_edges_per_relation=50_000):
    """Create a small edge-limited copy for quick forward-pass smoke tests."""
    small = HeteroData()

    for node_type in data.node_types:
        for key, value in data[node_type].items():
            small[node_type][key] = value

    for edge_type in data.edge_types:
        edge_index = data[edge_type].edge_index
        if edge_index.size(1) > max_edges_per_relation:
            edge_index = edge_index[:, :max_edges_per_relation]
        small[edge_type].edge_index = edge_index

    return small

## 4.6 - Type-Specific Feature Encoder

Each node type starts with a different input dimension. The encoder projects every node type into the same hidden dimension so message passing can operate in a shared latent space.

In [6]:
class HeteroFeatureEncoder(nn.Module):
    def __init__(self, metadata, input_dims, hidden_dim, dropout=0.0):
        super().__init__()
        self.node_types, _ = metadata
        self.dropout = nn.Dropout(dropout)

        self.encoders = nn.ModuleDict({
            node_type: nn.Sequential(
                nn.Linear(input_dims[node_type], hidden_dim),
                nn.LayerNorm(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            )
            for node_type in self.node_types
        })

    def forward(self, x_dict):
        return {
            node_type: self.encoders[node_type](x.float())
            for node_type, x in x_dict.items()
        }

## 4.7 - Heterogeneous Attention Layer

This is the core GNN layer.

For each relation, it computes multi-head attention from source nodes to destination nodes:

1. destination nodes produce queries
2. source nodes produce keys and values
3. attention scores are normalized over each destination node's incoming edges
4. weighted messages are summed into destination node updates
5. residual connections, normalization, and a feed-forward block stabilize the layer

This is GAT-style attention with HGT-style relation-specific projections.

In [7]:
class HeteroGraphAttentionLayer(nn.Module):
    def __init__(self, metadata, hidden_dim, num_heads=4, dropout=0.2):
        super().__init__()
        if hidden_dim % num_heads != 0:
            raise ValueError('hidden_dim must be divisible by num_heads')

        self.node_types, self.edge_types = metadata
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        self.scale = math.sqrt(self.head_dim)
        self.dropout = nn.Dropout(dropout)

        self.q_proj = nn.ModuleDict()
        self.k_proj = nn.ModuleDict()
        self.v_proj = nn.ModuleDict()
        self.out_proj = nn.ModuleDict()

        for edge_type in self.edge_types:
            key = self.edge_key(edge_type)
            self.q_proj[key] = nn.Linear(hidden_dim, hidden_dim, bias=False)
            self.k_proj[key] = nn.Linear(hidden_dim, hidden_dim, bias=False)
            self.v_proj[key] = nn.Linear(hidden_dim, hidden_dim, bias=False)
            self.out_proj[key] = nn.Linear(hidden_dim, hidden_dim, bias=False)

        self.node_norm_1 = nn.ModuleDict({
            node_type: nn.LayerNorm(hidden_dim)
            for node_type in self.node_types
        })
        self.node_norm_2 = nn.ModuleDict({
            node_type: nn.LayerNorm(hidden_dim)
            for node_type in self.node_types
        })
        self.ffn = nn.ModuleDict({
            node_type: nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim * 2),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim * 2, hidden_dim),
                nn.Dropout(dropout),
            )
            for node_type in self.node_types
        })

    @staticmethod
    def edge_key(edge_type):
        return '__'.join(edge_type)

    def forward(self, x_dict, edge_index_dict):
        messages = {
            node_type: x.new_zeros(x.shape)
            for node_type, x in x_dict.items()
        }

        for edge_type, edge_index in edge_index_dict.items():
            src_type, rel_type, dst_type = edge_type
            key = self.edge_key(edge_type)

            src, dst = edge_index
            x_src = x_dict[src_type]
            x_dst = x_dict[dst_type]

            q = self.q_proj[key](x_dst[dst]).view(-1, self.num_heads, self.head_dim)
            k = self.k_proj[key](x_src[src]).view(-1, self.num_heads, self.head_dim)
            v = self.v_proj[key](x_src[src]).view(-1, self.num_heads, self.head_dim)

            scores = (q * k).sum(dim=-1) / self.scale
            alpha = edge_softmax(scores, dst, x_dst.size(0))
            alpha = self.dropout(alpha)

            rel_messages = (alpha.unsqueeze(-1) * v).reshape(-1, self.hidden_dim)
            rel_messages = self.out_proj[key](rel_messages)

            messages[dst_type].index_add_(0, dst, rel_messages)

        out_dict = {}
        for node_type, x in x_dict.items():
            h = self.node_norm_1[node_type](x + self.dropout(messages[node_type]))
            h = self.node_norm_2[node_type](h + self.ffn[node_type](h))
            out_dict[node_type] = h

        return out_dict

## 4.8 - Full Fraud GNN

The full model stacks multiple heterogeneous attention layers and predicts fraud only for transaction nodes.

In [8]:
class RelationalFraudGNN(nn.Module):
    def __init__(self, metadata, input_dims, hidden_dim=64, num_heads=4, num_layers=2, dropout=0.25, num_classes=2):
        super().__init__()
        self.metadata = metadata
        self.encoder = HeteroFeatureEncoder(metadata, input_dims, hidden_dim, dropout)
        self.layers = nn.ModuleList([
            HeteroGraphAttentionLayer(metadata, hidden_dim, num_heads, dropout)
            for _ in range(num_layers)
        ])
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, data):
        x_dict = {node_type: data[node_type].x for node_type in data.node_types}
        edge_index_dict = {
            edge_type: data[edge_type].edge_index
            for edge_type in data.edge_types
        }

        h_dict = self.encoder(x_dict)
        for layer in self.layers:
            h_dict = layer(h_dict, edge_index_dict)

        transaction_logits = self.classifier(h_dict['transaction'])
        return transaction_logits, h_dict

    @torch.no_grad()
    def predict_proba(self, data):
        self.eval()
        logits, _ = self(data)
        return logits.softmax(dim=-1)

## 4.9 - Instantiate the Model

Input dimensions are read directly from the graph, so Phase 04 stays compatible if Phase 03 changes feature counts later.

In [9]:
metadata = data.metadata()
input_dims = {
    node_type: data[node_type].x.size(-1)
    for node_type in data.node_types
}

model = RelationalFraudGNN(
    metadata=metadata,
    input_dims=input_dims,
    hidden_dim=MODEL_CONFIG['hidden_dim'],
    num_heads=MODEL_CONFIG['num_heads'],
    num_layers=MODEL_CONFIG['num_layers'],
    dropout=MODEL_CONFIG['dropout'],
    num_classes=MODEL_CONFIG['num_classes'],
)

num_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model.__class__.__name__)
print(f'Total parameters     : {num_params:,}')
print(f'Trainable parameters : {trainable_params:,}')

RelationalFraudGNN
Total parameters     : 1,334,914
Trainable parameters : 1,334,914


## 4.10 - Forward-Pass Smoke Test

This checks that the model connects to the graph and returns transaction logits with shape `[num_transactions, 2]`.

For memory safety, the smoke test can limit the number of edges per relation. Phase 05 can decide whether to train full-batch or use neighbor sampling/subgraph training.

In [10]:
RUN_FULL_GRAPH_SMOKE_TEST = False
MAX_SMOKE_EDGES_PER_RELATION = 50_000

smoke_data = data if RUN_FULL_GRAPH_SMOKE_TEST else clone_with_limited_edges(
    data,
    max_edges_per_relation=MAX_SMOKE_EDGES_PER_RELATION,
)

model = model.to(DEVICE)
smoke_data = move_hetero_data(smoke_data, DEVICE)
model.eval()

with torch.no_grad():
    logits, embeddings = model(smoke_data)

print(f'Logits shape: {tuple(logits.shape)}')
print(f'Expected    : ({smoke_data["transaction"].num_nodes}, 2)')
print()
print('Embedding shapes:')
for node_type, h in embeddings.items():
    print(f'  {node_type:<15} {tuple(h.shape)}')

assert logits.shape == (smoke_data['transaction'].num_nodes, 2)
assert torch.isfinite(logits).all()
print()
print('Smoke test passed.')

Logits shape: (590540, 2)
Expected    : (590540, 2)

Embedding shapes:
  transaction     (590540, 64)
  product         (6, 64)
  card1           (6513, 64)
  card2           (501, 64)
  card3           (77, 64)
  card4           (6, 64)
  card5           (77, 64)
  card6           (6, 64)
  addr1           (134, 64)
  addr2           (31, 64)
  email_p         (61, 64)
  email_r         (62, 64)
  device_type     (4, 64)
  device_info     (889, 64)

Smoke test passed.


## 4.11 - Save an Untrained Model Checkpoint

This checkpoint stores the architecture config and randomly initialized weights. Phase 05 can load it before training, or simply re-create the model from this notebook's classes.

In [11]:
CHECKPOINT_PATH = DATA_DIR / 'phase04_untrained_model.pt'

checkpoint = {
    'model_state_dict': model.cpu().state_dict(),
    'model_config': MODEL_CONFIG,
    'input_dims': input_dims,
    'metadata': metadata,
    'graph_path': str(GRAPH_PATH),
}

torch.save(checkpoint, CHECKPOINT_PATH)
print(f'Saved untrained model checkpoint to: {CHECKPOINT_PATH}')

Saved untrained model checkpoint to: content\sample_data\phase04_untrained_model.pt


## 4.12 - What Phase 05 Should Do Next

Phase 05 should add:

- class-weighted cross entropy or focal loss for the 3.5% fraud imbalance
- optimizer and learning-rate schedule
- train/validation loop using `data['transaction'].train_mask` and `val_mask`
- early stopping on validation AUC-ROC or average precision
- final test evaluation using `test_mask`

The model's forward call returns transaction logits, so the training step will look like:

`logits, _ = model(data)`

then apply the loss only on transaction nodes selected by the mask.